### Hugging Face
- 이미지캡셔닝   :  Salesforce/blip-image-captioning-base
- 텍스트 임베딩  :  sentence-transformers/all-MiniLM-L6-v2
- 텍스트 생성    : distilbert/distilgpt2  (경량 모델)

In [1]:
import os
import fitz
import chromadb
from PIL import Image
from transformers import pipeline,BlipProcessor, BlipForConditionalGeneration
from chromadb.utils import embedding_functions

In [2]:
# BLIP 모델 로드
processor = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
model = BlipForConditionalGeneration.from_pretrained('Salesforce/blip-image-captioning-base')

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [3]:
# 임베딩 모델 로드
hf_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
# vectorDB 초기화
chroma_client = chromadb.PersistentClient(path='./chroma_db_hf')
collection = chroma_client.get_or_create_collection(name='multimocal_rag_hf',embedding_function=hf_ef)

### 데이터추출 및 오픈소스 임베딩
- PDF 텍스트와 추출된 이미지를 BLIP 모델로 갭셔닝한 뒤, Sentence Transformers임베딩을 사용해서 로컬 vectorDB에 저장

In [5]:
ptf_path = 'sample_paper.pdf'
if collection.count() == 0:
    doc = fitz.open(ptf_path)
    documents,metadatas,ids = [],[],[]
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        # 텍스트 추가
        text = page.get_text()
        if text.strip():
            documents.append(text)
            metadatas.append({'type':'text','page':page_num+1})
            ids.append(f'hf_text_page_{page_num+1}')
        # 이미지 캡셔닝
        image_lists = page.get_images(full=True)
        if image_lists:
            xref = image_lists[0][0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image['image']
            ext = base_image['ext']
            image_filename = f'ht_temp_img.{ext}'
            with open(image_filename, 'wb') as f:
                f.write(image_bytes)
            
            try:
                raw_image = Image.open(image_filename).convert('RGB')
                inputs = processor(raw_image, return_tensors='pt')
                out = model.generate(**inputs,max_new_tokens=50)
                caption = processor.decode(out[0],skip_special_tokens=True)

                documents.append(caption)
                metadatas.append({'type':'image_summary', 'page':page_num+1})
                ids.append(f'hf_image_summary_page_{page_num+1}')
                print(f'페이지 {page_num+1} 이미지 캡셔닝 완료 : {caption}')
            except Exception as e:
                print(f'페이지 {page_num+1} 이미지 캡셔닝 실패 : {e}')
    if documents:
        collection.add(documents=documents, metadatas=metadatas, ids=ids)
else:
    print(collection.count())


15


### RAG 검색 및 LLM을 통한 답변 생성

In [7]:
query = 'what is transformer architecture?'
results = collection.query(
    query_texts=[query],
    n_results=1
)
retrieved_docs = results['documents'][0]
retrieved_metas = results['metadatas'][0]
print(f'검색된 문서 수 : {len(retrieved_docs)}')
for meta in retrieved_metas:
    print(f" - [출처:페이지{meta['page']}, 타입] : {meta['type']}")
context = '\n\n--\n\n'.join(retrieved_docs) 
print('\n오픈소스 답변 생성중\n')
generator = pipeline('text-generation', model='distilbert/distilgpt2')
prompt = f'Context:{context[:400]}\n\nQuestion:{query}\n\nAnswer:'
response = generator(prompt,max_new_tokens=100, num_return_sequences=1)
print('최종답변')
print(response[0]['generated_text'].replace(prompt, "").strip())

검색된 문서 수 : 1
 - [출처:페이지3, 타입] : text

오픈소스 답변 생성중



config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Playdata\.cache\huggingface\hub\models--distilbert--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though

최종답변
a. The
Concept of the
transformer is that of
the
concept of the
decoder.
3.2
Decoder and Decoder Stacks
Encoder:
The decoder is composed of a stack of N = 6 identical layers.
The
decoder is composed of a stack of N = 6 identical layers.
The
decoder is composed of a stack of N = 6 identical layers.
The
decoder is composed of a stack


#  로컬 LLaVA 기반 고급 멀티모달 RAG

(경량 오픈소스 모델)을 넘어, ChatGPT-4o(Vision)에 필적하는 고성능 오픈소스 멀티모달 모델인 **LLaVA(Large Language-and-Vision Assistant)**를 로컬 환경에 세팅하고 활용하는  계획입니다. 

---

## 1. LLaVA 로컬 세팅이란? (What & Why)

- **LLaVA란?**: 언어 모델(예: Llama-2/3, Vicuna)과 비전 인코더(CLIP)를 결합하여 이미지와 텍스트를 동시에 이해하는 대규모 멀티모달 오픈소스 모델입니다.
- **도입 목적**: 
  - **보안(Privacy)**: 사내 기밀 문서나 고객 데이터(이미지, 영수증 등)를 OpenAI 서버로 전송하지 않고 100% 로컬에서 처리 가능합니다.
  - **성능**: `BLIP`과 같은 경량 캡셔닝 모델과 달리, 복잡한 인포그래픽, 다이어그램, 수학 공식이 포함된 이미지를 심도 있게 추론하고 분석할 수 있습니다.

---

## 2. 하드웨어 및 인프라 요구사항 (Requirements)

> [!WARNING]
> 고성능 멀티모달 LLM을 구동하기 위해서는 일정 수준 이상의 하드웨어가 필수적입니다.
- **권장 사양 (FP16 기준)**: VRAM 16GB~24GB 이상의 GPU (예: RTX 3090, 4090, A10G) 또는 고용량 통합 메모리를 갖춘 Apple Silicon (M2/M3 Max).
- **최소 사양 (4-bit 양자화 적용 시)**: VRAM 8GB 이상의 GPU (예: RTX 3060, 4060).
- **프레임워크**: `Ollama`(초보자/빠른 구축) 또는 `transformers` + `bitsandbytes` (파이썬 기반 세밀한 제어).

---

## 3. 단계별 구현 프로세스 (Proposed Plan)

### A: 로컬 인퍼런스 서버 구축 (Ollama 활용)
파이썬 패키지 의존성 충돌을 피하고 가장 안정적으로 LLaVA를 띄우는 방법을 학습합니다.
1. **Ollama 설치**: 로컬 PC에 Ollama 런타임을 설치합니다.
2. **LLaVA 모델 풀(Pull)**: 터미널에서 `ollama run llava:7b` (또는 13b) 명령어로 모델을 다운로드하고 구동을 테스트합니다.
3. **API 연동 테스트**: 파이썬에서 `langchain-community`의 `Ollama` 래퍼를 이용해 로컬 서버에 이미지를 보내고 답변을 받는 스크립트를 작성합니다.

### B: 고급 시각 임베딩 (Vision Embedding)
텍스트로 변환하여 저장하는 기존 방식을 넘어, **이미지 자체를 벡터화**하는 방식을 학습합니다.
1. **오픈소스 멀티모달 임베딩**: Nomic Vision (`nomic-embed-vision-v1.5`) 또는 OpenAI CLIP 모델을 사용합니다.
2. **이미지 벡터 DB 구축**: 크로마(Chroma) DB에 텍스트가 아닌 이미지의 픽셀 정보를 직접 임베딩하여 저장합니다.

### C: End-to-End 로컬 멀티모달 RAG 구축
1. **검색 (Retrieval)**: 사용자가 질문하면, 텍스트와 쿼리 이미지를 모두 활용하여 Vector DB에서 가장 유사한 '원본 이미지'와 '문서 단락'을 검색합니다.
2. **생성 (Generation)**: 검색된 텍스트와 **원본 이미지 자체(Base64)**를 로컬에 떠 있는 `LLaVA` 모델에 주입합니다.
3. **결과 분석**: LLaVA가 이미지를 직접 보고 답변한 내용과, Phase 3의 `BLIP + distilgpt2` 조합의 품질을 비교 분석합니다.

---

- RAG(이미지 인식->임베딩->검색->답변생성) 가능
- 모델 성능차이.. 경량화 모델인 BLIP는 문맥이해력이 낮아서 반복적인 텍스트를 출력
- 향후개선 : VRAM확보, 이미지인식모델은 LLaVA-1.5 이상,, 생성모델 Llama-3(8B) 수준으로 교체

In [ ]:
# 필수 패키지 설치
# pip install langchain-community langchain-core ollama requests Pillow

# https://ollama.com/download/windows


# C:\Users\Playdata>ollama --version
# ollama version is 0.30.7

# C:\Users\Playdata>ollama pull llava:7b
# pulling manifest
# pulling 170370233dd5: 100% ▕█████████████████████████████████████████████████████████████████▏ 4.1 GB
# pulling 72d6f08a42f6: 100% ▕█████████████████████████████████████████████████████████████████▏ 624 MB
# pulling 43070e2d4e53: 100% ▕█████████████████████████████████████████████████████████████████▏  11 KB
# pulling c43332387573: 100% ▕█████████████████████████████████████████████████████████████████▏   67 B
# pulling ed11eda7790d: 100% ▕█████████████████████████████████████████████████████████████████▏   30 B
# pulling 7c658f9561e5: 100% ▕█████████████████████████████████████████████████████████████████▏  564 B
# verifying sha256 digest
# writing manifest
# success

# C:\Users\Playdata>ollama run llava:7b
# >>> 안녕하세요 당신은 무엇을 할수 있나요?
#  안녕하세요! 제가 할 수 있는 일은 질문에 대해 답변을 제공하고, 특정한 주제에 대해 자세한 설명을 할 수 있습니다.

# >>> /bye

```
[사용자 질문]
     ↓
[텍스트 → CLIP 임베딩]          ← embedder.py
     ↓
[ChromaDB 유사 이미지 검색]      ← vector_store.py
     ↓
[검색된 이미지 → Base64 인코딩]  ← utils.py
     ↓
[이미지 + 질문 → LLaVA 주입]    ← Phase A 서버
     ↓
[최종 답변 생성]
```

```
# 1. 가상환경 활성화


# 2. Ollama 서버 실행 확인
# 트레이 아이콘 확인 또는:
curl http://localhost:11434

# 3. 대화형 RAG 실행 (메인)
python rag_chat.py

# 4. 품질 비교 벤치마크 (선택)
pip install transformers     # BLIP 필요 시 추가 설치
python benchmark.py
```

```
전체 파이프라인 완성 요약
Phase           핵심파일                               역할
A           utils.py, test_image.py             Ollama 서버 + LLaVA 기본 연동
B           embedder.py, vector_store.py        CLIP 임베딩 + ChromaDB 저장/검색    
C           rag_pipeline.py, rag_chat.py        검색-생성 통합 + 대화형 인터페이스
C           benchmark.py                        LLaVA vs BLIP 품질 비교
```